# 🛡️ Day 03a: Isolation Levels — Read Uncommitted to Serializable

---

## 🎯 What You'll Master Today
- All 4 SQL isolation levels
- Which anomalies each level prevents
- How databases implement each level
- PostgreSQL vs MySQL default differences

---

```
╔══════════════════════════════════════════════════════════════════════╗
║              ISOLATION LEVELS — ANOMALY PREVENTION                  ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Level                │ Dirty Read │ Non-Repeat │ Phantom │ Perf    ║
║  ─────────────────────┼────────────┼────────────┼─────────┼──────   ║
║  READ UNCOMMITTED     │  Possible  │  Possible  │ Possible│ Fastest ║
║  READ COMMITTED       │  Prevented │  Possible  │ Possible│ Fast    ║
║  REPEATABLE READ      │  Prevented │  Prevented │ Possible│ Medium  ║
║  SERIALIZABLE         │  Prevented │  Prevented │Prevented│ Slowest ║
║                                                                      ║
║  Defaults:                                                           ║
║  • PostgreSQL: READ COMMITTED                                       ║
║  • MySQL/InnoDB: REPEATABLE READ (with gap locks → no phantoms!)    ║
║  • SQL Server: READ COMMITTED                                       ║
║  • Oracle: READ COMMITTED (uses MVCC)                                ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Dirty Read Demo (READ UNCOMMITTED)
# ============================================

print("  === DIRTY READ (Read Uncommitted allows this) ===\n")

class IsolationDemo:
    def __init__(self):
        self.committed = {"price": 100}
        self.uncommitted = {}
    
    def dirty_read_scenario(self):
        print("  T1: BEGIN")
        print(f"  T1: UPDATE price = 50 (uncommitted)")
        self.uncommitted["price"] = 50
        
        print(f"  T2: READ price = {self.uncommitted.get('price', self.committed['price'])}")
        print(f"       ↑ T2 sees uncommitted value!")
        
        print(f"  T1: ROLLBACK")
        self.uncommitted.clear()
        print(f"  T2 used price=50 but it was rolled back! Price is still {self.committed['price']}")
        print(f"  ✗ DIRTY READ: T2 read data that never existed!")

demo = IsolationDemo()
demo.dirty_read_scenario()
print("\n  Fix: Use READ COMMITTED or higher — only read committed data")

In [ ]:
# ============================================
# 2. Non-Repeatable Read Demo (READ COMMITTED)
# ============================================

print("  === NON-REPEATABLE READ (Read Committed allows this) ===\n")

database = {"salary": 5000}

print("  T1: BEGIN")
read1 = database["salary"]
print(f"  T1: SELECT salary → {read1}")

print(f"  T2: UPDATE salary = 8000; COMMIT")
database["salary"] = 8000

read2 = database["salary"]
print(f"  T1: SELECT salary → {read2}  (same query, different result!)")
print(f"  ✗ NON-REPEATABLE: T1 got {read1} then {read2} for same row!")
print("\n  Fix: REPEATABLE READ — T1's snapshot frozen at BEGIN")

In [ ]:
# ============================================
# 3. Phantom Read Demo (REPEATABLE READ)
# ============================================

print("  === PHANTOM READ (Repeatable Read allows this in SQL standard) ===\n")

employees = [
    {"id": 1, "dept": "Engineering", "salary": 90000},
    {"id": 2, "dept": "Engineering", "salary": 85000},
]

print("  T1: SELECT COUNT(*) FROM employees WHERE dept='Engineering'")
count1 = len([e for e in employees if e["dept"] == "Engineering"])
print(f"      → {count1} rows")

print("  T2: INSERT INTO employees (id=3, dept='Engineering', salary=92000); COMMIT")
employees.append({"id": 3, "dept": "Engineering", "salary": 92000})

print("  T1: SELECT COUNT(*) FROM employees WHERE dept='Engineering'")
count2 = len([e for e in employees if e["dept"] == "Engineering"])
print(f"      → {count2} rows  (phantom row appeared!)")
print(f"  ✗ PHANTOM: Same query, {count1} then {count2} rows!")
print("\n  Fix: SERIALIZABLE — range locks or predicate locking")
print("  Note: MySQL InnoDB REPEATABLE READ uses gap locks → prevents most phantoms!")

In [ ]:
# ============================================
# 4. Isolation Level Implementation
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════════╗
  ║            HOW DATABASES IMPLEMENT ISOLATION LEVELS                  ║
  ╠══════════════════════════════════════════════════════════════════════╣
  ║                                                                      ║
  ║  READ UNCOMMITTED:                                                   ║
  │    → No read locks. Write locks only. Fastest but dirtiest.         ║
  ║                                                                      ║
  ║  READ COMMITTED:                                                     ║
  │    → Lock-based: S-lock acquired & released per statement.          ║
  │    → MVCC: Read latest committed snapshot per statement.             ║
  ║                                                                      ║
  ║  REPEATABLE READ:                                                    ║
  │    → Lock-based: S-locks held until txn ends.                       ║
  │    → MVCC: Read snapshot frozen at txn start (PostgreSQL).           ║
  │    → MySQL: gap locks prevent phantom inserts!                       ║
  ║                                                                      ║
  ║  SERIALIZABLE:                                                       ║
  │    → Lock-based: S/X locks + range locks (predicate locking).       ║
  │    → MVCC: Serializable Snapshot Isolation (SSI) — detects          ║
  │      read-write conflicts and aborts one txn.                        ║
  ║                                                                      ║
  ╚══════════════════════════════════════════════════════════════════════╝

  SQL to set isolation level:
  ┌─────────────────────────────────────────────────────────────┐
  │  SET TRANSACTION ISOLATION LEVEL READ COMMITTED;           │
  │  SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;          │
  │  SET TRANSACTION ISOLATION LEVEL SERIALIZABLE;             │
  │                                                             │
  │  -- PostgreSQL: per-session                                 │
  │  SET SESSION CHARACTERISTICS AS TRANSACTION                │
  │    ISOLATION LEVEL SERIALIZABLE;                            │
  └─────────────────────────────────────────────────────────────┘
""")

In [ ]:
# ============================================
# 5. Choosing the Right Isolation Level
# ============================================

print("""
  ┌─────────────────────┬──────────────────────────────────────────────┐
  │ Use Case             │ Recommended Level                           │
  ├─────────────────────┼──────────────────────────────────────────────┤
  │ Analytics / Reports  │ READ COMMITTED (tolerate minor staleness)  │
  │ E-commerce cart      │ READ COMMITTED (default, good enough)      │
  │ Banking transfers    │ SERIALIZABLE (correctness critical)        │
  │ Inventory management │ REPEATABLE READ (prevent lost updates)     │
  │ Counter / sequence   │ SERIALIZABLE or SELECT FOR UPDATE          │
  │ Read-heavy app       │ REPEATABLE READ with MVCC (no read blocks)│
  └─────────────────────┴──────────────────────────────────────────────┘

  Pro Tip: Most apps work fine with READ COMMITTED.
  Only use SERIALIZABLE when you NEED it — it's expensive.
  Prefer SELECT ... FOR UPDATE for specific critical sections.
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Name the 4 isolation levels? | Read Uncommitted → Read Committed → Repeatable Read → Serializable |
| 2 | What is a phantom read? | New rows appear in repeated range query due to another txn's INSERT |
| 3 | MySQL vs PostgreSQL default? | MySQL: REPEATABLE READ. PostgreSQL: READ COMMITTED |
| 4 | How does MVCC help? | Readers see snapshots, don't block writers. No lock contention on reads |
| 5 | When to use SERIALIZABLE? | Financial transactions, counter updates, any case needing strict ordering |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Higher isolation = fewer anomalies but slower         │
## │  • READ COMMITTED: default for most DBs (good enough)   │
## │  • REPEATABLE READ: MySQL default, uses gap locks       │
## │  • SERIALIZABLE: full correctness, use sparingly        │
## │  • MVCC: snapshot-based, avoids read-write blocking     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Deadlocks & Recovery** — Detection, Prevention, WAL Recovery